# Pandas for Python Developers

### Charles Rice - Senior Data Scientist
### Notebook available at https://github.com/dataville/pandas_for_python_developers
Website: https://www.datainference.ai

LinkedIn: https://www.linkedin.com/in/dataville

---

Pandas is a widely used Python library for working with tabular data.

**What this talk covers:**

- The pandas data model: Series, DataFrame, Index, dtypes
- Loading data and managing memory
- Selecting, filtering, sorting, and updating rows and columns
- Handling missing values and duplicates
- Grouping and aggregating with `groupby`
- Getting data back out into Python

In [1]:
import pandas as pd
import numpy as np
import random
import csv
from datetime import datetime, timedelta
from collections import Counter
import json

### The examples in this notebook require Pandas 3.x

In [2]:
pd.__version__

'3.0.2'

---
## Setup — Generate the Access Log

**The dataset:** Most examples use a synthetic web server access log with 5,000 rows, generated deterministically so every run produces the same results. 

Columns: timestamp, IP address, HTTP method, endpoint, status code, response time, bytes sent, and log level. 

The cell below generates the log and writes it to `access.log.csv` and `access.log.json`.

In [4]:
random.seed(42)

ENDPOINTS = [
    "/api/users", "/api/orders", "/api/products",
    "/api/auth/login", "/api/auth/logout",
    "/static/app.js", "/static/style.css",
    "/health", "/metrics",
]
IPS = [f"10.0.{random.randint(0,5)}.{random.randint(1,254)}" for _ in range(40)]
METHODS = ["GET"] * 7 + ["POST"] * 2 + ["DELETE"]

rows = []
t = datetime(2024, 3, 1, 0, 0, 0)
for _ in range(5000):
    t += timedelta(seconds=random.randint(1, 60))
    status = random.choices(
        [200, 201, 301, 400, 401, 404, 500, 503],
        weights=[60, 10, 5, 8, 5, 7, 4, 1]
    )[0]
    rows.append({
        "timestamp": t.strftime("%Y-%m-%d %H:%M:%S"),
        "ip":        random.choice(IPS),
        "method":    random.choice(METHODS),
        "endpoint":  random.choice(ENDPOINTS),
        "status_code": status,
        "response_ms": round(random.lognormvariate(4.5, 1.0)),
        "bytes_sent":  random.randint(200, 50000),
        "log_level": random.choices(["INFO", "WARN", "ERROR"], weights=[1, 994, 5])[0],
    })

with open("access.log.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

with open("access.log.json", "w") as f:
    json.dump(rows, f, indent=2)

print(f"Generated {len(rows)} rows → access.log.csv, access.log.json")


Generated 5000 rows → access.log.csv, access.log.json


## Section 1 — The Pandas Model

### Series

A `Series` is a one-dimensional array with an index. 

Think of it as a typed list where every element has a label.

A Series enforces a single dtype across all elements. 

If your data contains mixed types, Pandas will use the object dtype which is not memory efficent and lacks performance optimizations.


In [5]:
# From a plain Python list
response_times = pd.Series([120, 45, 890, 33, 210])
response_times

0    120
1     45
2    890
3     33
4    210
dtype: int64

In [6]:
# The index labels each row — 
# By default integers starting at 0
response_times.index.tolist()

[0, 1, 2, 3, 4]

### DataFrame

A `DataFrame` is a table. 

Under the hood it's a dict of Series.


In [7]:
# From a list of dicts, the shape you get from JSON / API responses
records = [
    {"endpoint": "/api/users",  "status_code": 200, "response_ms": 120},
    {"endpoint": "/api/orders", "status_code": 404, "response_ms": 45},
    {"endpoint": "/health",     "status_code": 200, "response_ms": 8},
]
pd.DataFrame(records)


,endpoint,status_code,response_ms
0,/api/users,200,120
1,/api/orders,404,45
2,/health,200,8


In [8]:
# From a dict of lists, column names are the keys
df_small = pd.DataFrame({
    "endpoint":    ["/api/users", "/api/orders", "/health",  "/api/products", "/metrics"],
    "status_code": [200, 404, 200, 500, 301],
    "response_ms": [120, 45, 8, 890, 33],
})
df_small


,endpoint,status_code,response_ms
0,/api/users,200,120
1,/api/orders,404,45
2,/health,200,8
3,/api/products,500,890
4,/metrics,301,33


### The Index

Both Series and DataFrames have an index. 

By default those labels are integers starting at 0. 

The index travels with each row through filtering and sorting.

The index does not have to be an integer; it can be strings, timestamps, or any hashable value.


In [9]:
df_small.index.tolist()

[0, 1, 2, 3, 4]

In [10]:
# An index can be strings, not just integers
endpoint_response = pd.Series(
    [120, 45, 8],
    index=["/api/users", "/api/orders", "/health"]
)
endpoint_response

/api/users     120
/api/orders     45
/health          8
dtype: int64

### dtypes

Every column has a dtype. Pandas infers it on creation, but what you get depends on both the data and how the column was built, and the result is not always what you would predict.

- An integer column with a missing value will have a dtype of `float64`.

- A Python list with mixed types such as `[1, 2, "oops", 4]` constructed in memory gets dtype object. Read those same values from a CSV and you get StringDtype instead. Same values, different dtype depending on the source.

Use `dtypes` to check what you actually got.

In [11]:
df = pd.DataFrame({"col": [1, 2, "oops", 4]})
df.dtypes


col    object
dtype: object

---
## Section 2 — Loading Data

Pandas has a family of `read_*` functions: `read_csv`, `read_json`, `read_excel`, `read_parquet`, `read_sql` (via SQLAlchemy); with a consistent interface across formats. 

Here is how reading a CSV compares to plain Python.


In [12]:
with open("access.log.csv") as f:
    rows = list(csv.DictReader(f))

rows[:5]

[{'timestamp': '2024-03-01 00:00:15',
  'ip': '10.0.0.98',
  'method': 'GET',
  'endpoint': '/health',
  'status_code': '401',
  'response_ms': '130',
  'bytes_sent': '24460',
  'log_level': 'WARN'},
 {'timestamp': '2024-03-01 00:00:58',
  'ip': '10.0.5.27',
  'method': 'DELETE',
  'endpoint': '/api/products',
  'status_code': '200',
  'response_ms': '97',
  'bytes_sent': '30494',
  'log_level': 'WARN'},
 {'timestamp': '2024-03-01 00:01:58',
  'ip': '10.0.1.181',
  'method': 'GET',
  'endpoint': '/static/app.js',
  'status_code': '201',
  'response_ms': '58',
  'bytes_sent': '14026',
  'log_level': 'WARN'},
 {'timestamp': '2024-03-01 00:02:35',
  'ip': '10.0.3.88',
  'method': 'GET',
  'endpoint': '/health',
  'status_code': '401',
  'response_ms': '11',
  'bytes_sent': '30271',
  'log_level': 'WARN'},
 {'timestamp': '2024-03-01 00:02:44',
  'ip': '10.0.1.181',
  'method': 'POST',
  'endpoint': '/api/auth/logout',
  'status_code': '200',
  'response_ms': '189',
  'bytes_sent': '38442',

Here's the same thing in pandas.


In [13]:
df = pd.read_csv("access.log.csv")
df.head()    # returns the first 5 rows; accepts an optional n argument, e.g. head(10)

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
0,2024-03-01 00:00:15,10.0.0.98,GET,/health,401,130,24460,WARN
1,2024-03-01 00:00:58,10.0.5.27,DELETE,/api/products,200,97,30494,WARN
2,2024-03-01 00:01:58,10.0.1.181,GET,/static/app.js,201,58,14026,WARN
3,2024-03-01 00:02:35,10.0.3.88,GET,/health,401,11,30271,WARN
4,2024-03-01 00:02:44,10.0.1.181,POST,/api/auth/logout,200,189,38442,WARN


The same interface reads JSON:


In [14]:
df = pd.read_json("access.log.json")
df.head()

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
0,2024-03-01 00:00:15,10.0.0.98,GET,/health,401,130,24460,WARN
1,2024-03-01 00:00:58,10.0.5.27,DELETE,/api/products,200,97,30494,WARN
2,2024-03-01 00:01:58,10.0.1.181,GET,/static/app.js,201,58,14026,WARN
3,2024-03-01 00:02:35,10.0.3.88,GET,/health,401,11,30271,WARN
4,2024-03-01 00:02:44,10.0.1.181,POST,/api/auth/logout,200,189,38442,WARN


### Memory

Pandas defaults to `int64` and `float64` for numeric columns. 

On a large dataset that's often more than you need.

In [15]:
df.dtypes

timestamp      datetime64[us]
ip                        str
method                    str
endpoint                  str
status_code             int64
response_ms             int64
bytes_sent              int64
log_level                 str
dtype: object

In [16]:
df.memory_usage(deep=True).sum() / 1024**2  # MB

np.float64(1.2271003723144531)

Cast columns to appropriate types. 

Status codes fit in `int16`. 

category is Pandas' representation of a column with a small, fixed set of repeated values, like HTTP methods or status codes.

Internally it stores an integer code per row and a single lookup table of the actual values. 

For a column with 5 distinct strings repeated across 5,000 rows, that's 5 strings stored once instead of 5,000.


In [17]:
df = df.astype({
    "status_code": "int16",
    "bytes_sent":  "int32",
    "response_ms": "int32",
    "endpoint":    "category",
    "method":      "category",
    "log_level":   "category",
})

Casting to smaller types reduces memory consumption to roughly one-third of the default allocation.

In [18]:
df.memory_usage(deep=True).sum() / 1024**2  # MB after casting

np.float64(0.37990665435791016)

In [19]:
df.dtypes

timestamp      datetime64[us]
ip                        str
method               category
endpoint             category
status_code             int16
response_ms             int32
bytes_sent              int32
log_level            category
dtype: object

You can also specify dtypes at read time by passing a `dtype` mapping to `read_csv`/`read_json`. This is more memory-efficient than casting after the fact, because Pandas reads the data straight into the smaller types instead of allocating the larger `int64`/`float64` columns first and then converting.


Even with these optimizations, the data may not fit in memory. 

Dask, Modin, Ray, Vaex, cuDF, and PySpark provide pandas compatible APIs that can handle larger-than-memory datasets; some on a single machine, others across a cluster. Those are outside the scope of this talk.

---
## Section 3 — Core Operations

This section covers the operations you will use in almost every Pandas workflow: selecting columns and rows, filtering on conditions, updating values, sorting, handling missing data and duplicates, computing new columns with `np.where`, and grouping with `groupby`. 

It ends with the common exits, how to get data back out of a DataFrame and into Python.


### Selecting Rows and Columns

**Selecting columns** uses `df["col"]` for a single column (returns a Series) or `df[["col1", "col2"]]` for multiple columns (returns a DataFrame). The double brackets are not a typo, the inner list is the column selector.

`loc` and `iloc` select rows, and each accepts an optional second argument to select columns at the same time.

`loc` selects by label — the index value for rows, the column name for columns. Watch out for one difference from Python slicing: `loc[0:4]` returns every row whose label falls between 0 and 4 **inclusive**; five rows, not four.

`iloc` ignores the index and selects by position — zero-based, exclusive end, the same rules as Python list slicing.

In [20]:
# Single column by name
df["endpoint"].head()

0             /health
1       /api/products
2      /static/app.js
3             /health
4    /api/auth/logout
Name: endpoint, dtype: category
Categories (9, str): ['/api/auth/login', '/api/auth/logout', '/api/orders', '/api/products', ..., '/health', '/metrics', '/static/app.js', '/static/style.css']

In [21]:
# Multiple columns
df[["endpoint", "status_code", "response_ms"]].head()

,endpoint,status_code,response_ms
0,/health,401,130
1,/api/products,200,97
2,/static/app.js,201,58
3,/health,401,11
4,/api/auth/logout,200,189


In [22]:
# loc — rows by index label, columns by name
df.loc[0:4, ["endpoint", "status_code"]]

,endpoint,status_code
0,/health,401
1,/api/products,200
2,/static/app.js,201
3,/health,401
4,/api/auth/logout,200


In [23]:
# iloc — rows and columns by integer position
df.iloc[0:5, 0:3]

,timestamp,ip,method
0,2024-03-01 00:00:15,10.0.0.98,GET
1,2024-03-01 00:00:58,10.0.5.27,DELETE
2,2024-03-01 00:01:58,10.0.1.181,GET
3,2024-03-01 00:02:35,10.0.3.88,GET
4,2024-03-01 00:02:44,10.0.1.181,POST


### Common Operations

Once you have a column, these Series methods come up constantly.

`value_counts()` returns each distinct value paired with how often it appears, sorted from most frequent to least, the Pandas equivalent of `collections.Counter(...).most_common()`.

`min()`, `max()`, and `std()` return the smallest value, the largest value, and the standard deviation. Numeric columns only.

In [24]:
# Frequency of each distinct value
df["endpoint"].value_counts()

endpoint
/static/style.css    608
/metrics             578
/api/users           563
/health              560
/api/auth/login      551
/api/auth/logout     541
/api/orders          539
/api/products        533
/static/app.js       527
Name: count, dtype: int64

In [25]:
df["response_ms"].min()

np.int32(2)

In [26]:
df["response_ms"].max()

np.int32(4928)

In [27]:
df["response_ms"].std()

np.float64(200.16423201949942)

### Operations Return New DataFrames

Most Pandas methods return a new DataFrame and leave the original unchanged. 

To keep a transformed result, rebind the name. Below, `sort_values` returns a sorted result, but the underlying `df` is unchanged until we assign the result back.


In [28]:
df = pd.read_csv("access.log.csv")

In [29]:
# sort_values returns a sorted DataFrame
df.sort_values("response_ms").head(3)


,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
690,2024-03-01 05:54:23,10.0.1.246,GET,/static/app.js,201,2,35172,WARN
1910,2024-03-01 16:23:13,10.0.0.8,GET,/static/style.css,404,2,49541,WARN
396,2024-03-01 03:20:32,10.0.4.23,GET,/api/auth/login,301,3,17509,WARN


In [30]:
# but df itself is unchanged — head(3) is still in the original order
df.head(3)

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
0,2024-03-01 00:00:15,10.0.0.98,GET,/health,401,130,24460,WARN
1,2024-03-01 00:00:58,10.0.5.27,DELETE,/api/products,200,97,30494,WARN
2,2024-03-01 00:01:58,10.0.1.181,GET,/static/app.js,201,58,14026,WARN


In [31]:
# Rebind df to the sorted result — now the sort sticks
df = df.sort_values("response_ms")
df.head(3)


,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
690,2024-03-01 05:54:23,10.0.1.246,GET,/static/app.js,201,2,35172,WARN
1910,2024-03-01 16:23:13,10.0.0.8,GET,/static/style.css,404,2,49541,WARN
396,2024-03-01 03:20:32,10.0.4.23,GET,/api/auth/login,301,3,17509,WARN


### Filtering

Filtering rows is built on a boolean mask, a Series of True/False values, one per row, produced by applying a condition to a column. 

Passing that mask back into `df[]` keeps only the rows where the condition is True.

The inner expression evaluates first: `df["status_code"] >= 500` produces the mask. 

The outer `df[mask]` does the filtering.


In [32]:
df = pd.read_csv("access.log.csv")

In [33]:
# The mask
df["status_code"] >= 500      # Series of True/False

0       False
1       False
2       False
3       False
4       False
        ...  
4995     True
4996    False
4997    False
4998    False
4999    False
Name: status_code, Length: 5000, dtype: bool

In [34]:
# The filter
df[df["status_code"] >= 500]     # rows where status_code is 500 or above

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
5,2024-03-01 00:02:59,10.0.0.8,POST,/health,503,15,44796,WARN
19,2024-03-01 00:10:45,10.0.5.29,GET,/api/auth/login,503,30,14208,WARN
21,2024-03-01 00:12:28,10.0.1.181,POST,/api/products,500,46,4032,WARN
44,2024-03-01 00:25:35,10.0.1.130,GET,/api/auth/logout,500,97,22166,WARN
54,2024-03-01 00:31:29,10.0.2.155,POST,/metrics,500,88,32027,WARN
...,...,...,...,...,...,...,...,...
4894,2024-03-02 17:24:50,10.0.2.253,GET,/api/auth/login,503,399,8890,WARN
4896,2024-03-02 17:26:07,10.0.0.8,GET,/api/auth/logout,500,38,10897,WARN
4908,2024-03-02 17:32:00,10.0.0.187,DELETE,/health,500,144,44535,WARN
4928,2024-03-02 17:41:20,10.0.2.207,GET,/metrics,500,148,49217,WARN


### query()

Filters rows using a string expression. 

It's often more readable than boolean masks when you're combining multiple conditions.


In [35]:
# Using a boolean mask
df[(df["status_code"] >= 500) & (df["response_ms"] > 500)].head()

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
1107,2024-03-01 09:35:34,10.0.2.63,POST,/metrics,500,587,18201,WARN
1367,2024-03-01 11:43:49,10.0.4.109,POST,/api/products,500,576,28442,WARN
1462,2024-03-01 12:33:19,10.0.5.167,GET,/api/users,500,894,22570,WARN
1789,2024-03-01 15:22:23,10.0.3.57,POST,/api/auth/login,503,578,26306,WARN
3023,2024-03-02 01:30:31,10.0.5.167,POST,/api/auth/login,500,600,29952,WARN


In [36]:
# Same result with query()
df.query("status_code >= 500 and response_ms > 500").head()

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
1107,2024-03-01 09:35:34,10.0.2.63,POST,/metrics,500,587,18201,WARN
1367,2024-03-01 11:43:49,10.0.4.109,POST,/api/products,500,576,28442,WARN
1462,2024-03-01 12:33:19,10.0.5.167,GET,/api/users,500,894,22570,WARN
1789,2024-03-01 15:22:23,10.0.3.57,POST,/api/auth/login,503,578,26306,WARN
3023,2024-03-02 01:30:31,10.0.5.167,POST,/api/auth/login,500,600,29952,WARN


To reference a Python variable inside a query string, prefix it with @.

In [37]:
threshold = 500
df.query("response_ms > @threshold").head()

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
47,2024-03-01 00:28:07,10.0.2.207,GET,/api/products,200,501,30644,WARN
84,2024-03-01 00:46:13,10.0.3.88,DELETE,/api/auth/logout,200,768,38677,WARN
114,2024-03-01 01:00:48,10.0.2.148,GET,/api/products,404,612,32269,WARN
136,2024-03-01 01:10:57,10.0.5.167,GET,/api/auth/login,200,629,19169,WARN
163,2024-03-01 01:26:12,10.0.0.8,POST,/health,201,737,1445,WARN


### Assignment — loc and iloc

loc and iloc work for writing as well as reading. 

Put a loc or iloc call on the left-hand side of the assignment, and Pandas updates the DataFrame in place.

A common mistake is chained indexing: filtering with `df[mask]` and then setting a column on the result. 

In Pandas 3.x this emits a `ChainedAssignmentError` warning, the assignment is silently dropped, and the original DataFrame is unchanged. (iloc-based positional assignment to a single cell — e.g. `df.iloc[0, 3] = "X"` — works the same way as loc.)

AI coding tools produce this antipattern routinely.

The fix is to do the row selection and the column selection in a single `.loc` call:


In [38]:
df = pd.read_csv("access.log.csv")

# Chained indexing — the antipattern. Pandas 3.x prints a ChainedAssignmentError
# warning and the DataFrame is unchanged.
df[df["status_code"] == 200]["endpoint"] = "CHANGED"
df.loc[df["status_code"] == 200, "endpoint"].head(3)

/var/folders/qg/s6tmww6s0zv7vs41wx8t_07m0000gn/T/ipykernel_2078/1370520911.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df[df["status_code"] == 200]["endpoint"] = "CHANGED"


1       /api/products
4    /api/auth/logout
6            /metrics
Name: endpoint, dtype: str

In [39]:
# Reset, then show the "before" state with the correct single-call query
df = pd.read_csv("access.log.csv")
df.loc[df["status_code"] == 200, "endpoint"].head(3)

1       /api/products
4    /api/auth/logout
6            /metrics
Name: endpoint, dtype: str

In [40]:
# Assign with a single .loc call, then read the same query to see the result
df.loc[df["status_code"] == 200, "endpoint"] = "FOUND"
df.loc[df["status_code"] == 200, "endpoint"].head(3)

1    FOUND
4    FOUND
6    FOUND
Name: endpoint, dtype: str

### Sorting

Pass a column name and `ascending=False` to sort descending. 

The index retains its original labels, row 4661 is still row 4661, it has just moved to the top.


In [41]:
# Single column
df.sort_values("response_ms", ascending=False).head(5)

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
4661,2024-03-02 15:27:29,10.0.1.181,GET,/metrics,404,4928,40579,WARN
3661,2024-03-02 06:42:01,10.0.2.155,GET,/metrics,401,3172,34124,WARN
2439,2024-03-01 20:36:18,10.0.5.27,GET,/api/products,401,3073,1171,WARN
236,2024-03-01 02:05:37,10.0.0.56,POST,FOUND,200,2496,36722,WARN
1035,2024-03-01 08:54:12,10.0.0.250,GET,/api/orders,400,2451,18450,WARN


Pass a list of column names to sort by multiple columns, and a matching list of booleans to control direction independently. Here, rows are grouped by `log_level` alphabetically (`ERROR` → `INFO` → `WARN`), then within each group sorted by `response_ms` from slowest to fastest.

In [42]:
# Multi-column — slowest requests per log level
# Primary: log_level ascending (ERROR → INFO → WARN); secondary: response_ms descending
df.sort_values(["log_level", "response_ms"], ascending=[True, False]).head(20)


,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
1751,2024-03-01 15:00:38,10.0.5.159,POST,FOUND,200,366,49779,ERROR
213,2024-03-01 01:53:13,10.0.2.63,GET,FOUND,200,199,2581,ERROR
1459,2024-03-01 12:31:42,10.0.4.51,GET,FOUND,200,189,34162,ERROR
4842,2024-03-02 16:57:26,10.0.4.109,GET,FOUND,200,156,48421,ERROR
3628,2024-03-02 06:24:35,10.0.3.151,GET,/api/users,400,116,20873,ERROR
1991,2024-03-01 17:01:58,10.0.2.207,POST,FOUND,200,109,47238,ERROR
2613,2024-03-01 22:04:48,10.0.0.8,DELETE,FOUND,200,98,41941,ERROR
949,2024-03-01 08:05:34,10.0.4.76,GET,FOUND,200,90,44309,ERROR
4576,2024-03-02 14:41:20,10.0.5.190,POST,FOUND,200,77,32946,ERROR
3539,2024-03-02 05:44:11,10.0.4.51,GET,FOUND,200,62,41574,ERROR


### Missing Values

You don't always know whether a dataset has nulls, or which columns they're in. 

isnull() returns a DataFrame of True/False values with the same shape as the original, and summing over it gives a null count per column, zero where the column is clean, nonzero where it isn't.

What to do about nulls is handled on a case by case basis; there are no set rules as to what to do. 

A small percentage of nulls is usually safe to drop; but not always. 

When a large fraction of a column is null, dropping rows throws away too much data, and filling the nulls in is often the better option; though picking a sensible fill value is its own problem.


In [43]:
# Pandas represents missing numeric values as np.nan (NumPy's not-a-number).
# Reload from CSV so the index is a clean 0..4999 RangeIndex before injecting nulls.
# (df was sorted in-place earlier, which would have made loc[2:5] an empty slice.)
df = pd.read_csv("access.log.csv")
df_with_nulls = df.copy()
df_with_nulls.loc[2:5, "response_ms"] = np.nan
df_with_nulls.head(8)


,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
0,2024-03-01 00:00:15,10.0.0.98,GET,/health,401,130.0,24460,WARN
1,2024-03-01 00:00:58,10.0.5.27,DELETE,/api/products,200,97.0,30494,WARN
2,2024-03-01 00:01:58,10.0.1.181,GET,/static/app.js,201,NaN,14026,WARN
3,2024-03-01 00:02:35,10.0.3.88,GET,/health,401,NaN,30271,WARN
4,2024-03-01 00:02:44,10.0.1.181,POST,/api/auth/logout,200,NaN,38442,WARN
5,2024-03-01 00:02:59,10.0.0.8,POST,/health,503,NaN,44796,WARN
6,2024-03-01 00:03:04,10.0.2.253,POST,/metrics,200,35.0,952,WARN
7,2024-03-01 00:03:12,10.0.2.148,GET,/static/app.js,201,28.0,29935,WARN


dropna() returns a new DataFrame with rows containing nulls removed; the original is unchanged.

By default, how='any' drops a row if any column in the row contains a null. 

how='all' drops a row only when every column in the row is null.

Use subset= to limit the check to specific columns.

In [44]:
df_with_nulls.isnull().sum()


timestamp      0
ip             0
method         0
endpoint       0
status_code    0
response_ms    4
bytes_sent     0
log_level      0
dtype: int64

In [45]:
df_with_nulls.shape

(5000, 8)

In [46]:
# Drop any row that has a null in any column
df_with_nulls.dropna().shape


(4996, 8)

In [47]:
# Drop rows where response_ms is null
cleaned = df_with_nulls.dropna(subset=["response_ms"])

# The index now has a gap at rows 2-5
cleaned.index[:10].tolist()

[0, 1, 6, 7, 8, 9, 10, 11, 12, 13]

After `dropna()`, the index retains its original labels and has gaps. 

`reset_index(drop=True)` restores the index to `0..n`. Without `drop=True`, the old index is added as a column in the returned DataFrame instead of being discarded.


In [48]:
# reset_index gives you a clean 0..n RangeIndex
# drop=True discards the old index instead of making it a column
cleaned = cleaned.reset_index(drop=True)
cleaned.index[:10].tolist()


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

`fillna()` replaces nulls with a value you supply. 

Use a fixed value when a missing value has a known value or where a default value is appropriate. 

Use a computed value like the column median when you want to preserve the distribution without skewing it.


In [49]:
df_with_nulls["response_ms"].head()

0    130.0
1     97.0
2      NaN
3      NaN
4      NaN
Name: response_ms, dtype: float64

In [50]:
# Fill nulls with a fixed value
df_with_nulls["response_ms"].fillna(0).head()


0    130.0
1     97.0
2      0.0
3      0.0
4      0.0
Name: response_ms, dtype: float64

In [51]:
# Use the median as the fill value — preserves the column's distribution
median_response = df_with_nulls["response_ms"].median()
print(f"median = {median_response}")
df_with_nulls["response_ms"].fillna(median_response).head()


median = 90.0


0    130.0
1     97.0
2     90.0
3     90.0
4     90.0
Name: response_ms, dtype: float64

### Deduplication — drop_duplicates

`drop_duplicates()` returns a new DataFrame with duplicate rows removed; the original is unchanged.

By default it compares every column. 

Use `subset=` to limit the comparison to specific columns, a row is treated as a duplicate if those columns match, regardless of the other columns.

Like `dropna`, it leaves index gaps; use `reset_index(drop=True)` to restore index.

In [52]:
# Inject duplicates so there is something to remove
# pd.concat stacks a list of DataFrames row-wise into a single DataFrame
df_with_dupes = pd.concat([df.head(5), df.head(3)]).reset_index(drop=True)
df_with_dupes

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
0,2024-03-01 00:00:15,10.0.0.98,GET,/health,401,130,24460,WARN
1,2024-03-01 00:00:58,10.0.5.27,DELETE,/api/products,200,97,30494,WARN
2,2024-03-01 00:01:58,10.0.1.181,GET,/static/app.js,201,58,14026,WARN
3,2024-03-01 00:02:35,10.0.3.88,GET,/health,401,11,30271,WARN
4,2024-03-01 00:02:44,10.0.1.181,POST,/api/auth/logout,200,189,38442,WARN
5,2024-03-01 00:00:15,10.0.0.98,GET,/health,401,130,24460,WARN
6,2024-03-01 00:00:58,10.0.5.27,DELETE,/api/products,200,97,30494,WARN
7,2024-03-01 00:01:58,10.0.1.181,GET,/static/app.js,201,58,14026,WARN


`duplicated()` returns a boolean Series, `True` for every row that is an exact copy of an earlier row. By default (`keep='first'`) the first occurrence is treated as the original; every subsequent copy is flagged.


In [53]:
df_with_dupes.duplicated()

0    False
1    False
2    False
3    False
4    False
5     True
6     True
7     True
dtype: bool

Use the boolean Series as a mask to display the actual duplicate rows.

In [54]:
df_with_dupes[df_with_dupes.duplicated()]

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
5,2024-03-01 00:00:15,10.0.0.98,GET,/health,401,130,24460,WARN
6,2024-03-01 00:00:58,10.0.5.27,DELETE,/api/products,200,97,30494,WARN
7,2024-03-01 00:01:58,10.0.1.181,GET,/static/app.js,201,58,14026,WARN


`drop_duplicates()` removes the duplicate rows and returns a new DataFrame; the original is unchanged. 

`reset_index(drop=True)` restores the index to 0..n.

In [55]:
deduped = df_with_dupes.drop_duplicates().reset_index(drop=True)
deduped

,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
0,2024-03-01 00:00:15,10.0.0.98,GET,/health,401,130,24460,WARN
1,2024-03-01 00:00:58,10.0.5.27,DELETE,/api/products,200,97,30494,WARN
2,2024-03-01 00:01:58,10.0.1.181,GET,/static/app.js,201,58,14026,WARN
3,2024-03-01 00:02:35,10.0.3.88,GET,/health,401,11,30271,WARN
4,2024-03-01 00:02:44,10.0.1.181,POST,/api/auth/logout,200,189,38442,WARN


In [56]:
print(f"Before: {len(df_with_dupes)} rows")
print(f"After:  {len(deduped)} rows")

Before: 8 rows
After:  5 rows


### Adding Rows

There is no in-place row append; every row-addition operation returns a new DataFrame. 

The `df.append()` method from older versions of Pandas has been removed.

Coding agents will often try and use append which will produce an error.

**For incremental data** (a loop, a queue, an API response stream) the correct pattern is to accumulate records in a plain Python list and call `pd.DataFrame()` once when the batch is complete. 

Calling pd.concat() inside a loop copies the entire DataFrame on every iteration.

To add a block of rows to an existing DataFrame, pass both as a list to `pd.concat()` with `ignore_index=True`. 

Without `ignore_index=True` the new rows keep their original index values, which duplicates indices already in the existing DataFrame.

In [57]:
# Simulate a batch of incoming log entries — from a queue, socket, or API.
# Accumulate plain dicts in a Python list, then build the DataFrame once.
incoming = [
    {"timestamp": "2024-03-15 10:00:01", "ip": "10.0.1.5",  "method": "GET",  "endpoint": "/api/users",  "status_code": 200, "response_ms": 42,  "bytes_sent": 2048, "log_level": "INFO"},
    {"timestamp": "2024-03-15 10:00:05", "ip": "10.0.2.10", "method": "POST", "endpoint": "/api/orders", "status_code": 201, "response_ms": 88,  "bytes_sent": 1024, "log_level": "INFO"},
    {"timestamp": "2024-03-15 10:00:09", "ip": "10.0.0.3",  "method": "GET",  "endpoint": "/health",     "status_code": 500, "response_ms": 300, "bytes_sent": 512,  "log_level": "ERROR"},
]

new_rows = pd.DataFrame(incoming)
new_rows


,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
0,2024-03-15 10:00:01,10.0.1.5,GET,/api/users,200,42,2048,INFO
1,2024-03-15 10:00:05,10.0.2.10,POST,/api/orders,201,88,1024,INFO
2,2024-03-15 10:00:09,10.0.0.3,GET,/health,500,300,512,ERROR


In [58]:
# Extend the existing DataFrame with the new rows.
# ignore_index=True renumbers the index from 0 so there are no duplicates.
combined = pd.concat([df, new_rows], ignore_index=True)
print(f"df:       {len(df)} rows")
print(f"new_rows: {len(new_rows)} rows")
print(f"combined: {len(combined)} rows")
combined.tail()


df:       5000 rows
new_rows: 3 rows
combined: 5003 rows


,timestamp,ip,method,endpoint,status_code,response_ms,bytes_sent,log_level
4998,2024-03-02 18:16:29,10.0.2.27,GET,/metrics,200,56,29971,WARN
4999,2024-03-02 18:17:10,10.0.5.29,GET,/static/app.js,401,229,44957,WARN
5000,2024-03-15 10:00:01,10.0.1.5,GET,/api/users,200,42,2048,INFO
5001,2024-03-15 10:00:05,10.0.2.10,POST,/api/orders,201,88,1024,INFO
5002,2024-03-15 10:00:09,10.0.0.3,GET,/health,500,300,512,ERROR


### Derived Columns — np.where()

`np.where(condition, value_if_true, value_if_false)` evaluates a boolean condition across an entire column and returns an array; no Python loop.

Two common uses:
- **Add a derived column** — assign the result to a new column name
- **Modify an existing column** — pass the original column as `value_if_false` so only matching rows change


In [59]:
df = pd.read_csv("access.log.csv")

# Add a derived column — two outcomes, one condition
df["speed"] = np.where(df["response_ms"] > 100, "slow", "fast")
df[["response_ms", "speed"]].head(10)


,response_ms,speed
0,130,slow
1,97,fast
2,58,fast
3,11,fast
4,189,slow
5,15,fast
6,35,fast
7,28,fast
8,91,fast
9,239,slow


In [60]:
# Check the maximum response time
df["response_ms"].max()

np.int64(4928)

In [61]:
# Modify an existing column — cap response_ms at 500
# Rows where response_ms > 500 get 500; all others keep their original value
df["response_ms"] = np.where(df["response_ms"] > 500, 500, df["response_ms"])
df["response_ms"].max()


np.int64(500)

### groupby

`groupby` splits a DataFrame into groups based on the values in a column. 

It returns a `GroupBy` object; no computation has happened yet. 

You call a method on it to produce a result.

The most common use is aggregation: reduce each group down to a single summary row. 

In [62]:
# Reload to undo the response_ms cap from the np.where example
df = pd.read_csv("access.log.csv")

In [63]:
# groupby() returns a GroupBy object — no computation has happened yet
g = df.groupby("endpoint")
type(g)

pandas.api.typing.DataFrameGroupBy

In [64]:
# Call an aggregation method on the GroupBy object to produce a result
g["response_ms"].mean()

endpoint
/api/auth/login      146.647913
/api/auth/logout     155.937153
/api/orders          136.009276
/api/products        165.849906
/api/users           153.003552
/health              150.344643
/metrics             149.747405
/static/app.js       144.265655
/static/style.css    134.740132
Name: response_ms, dtype: float64

#### The group key becomes the index

The group key becomes the **index** of the result.

Call `.reset_index()` to turn the key back into a regular column, or pass `as_index=False` to `groupby` to do it in one step.


In [65]:
# Option 1: call .reset_index() on the result
df.groupby("endpoint")["response_ms"].mean().reset_index()

,endpoint,response_ms
0,/api/auth/login,146.647913
1,/api/auth/logout,155.937153
2,/api/orders,136.009276
3,/api/products,165.849906
4,/api/users,153.003552
5,/health,150.344643
6,/metrics,149.747405
7,/static/app.js,144.265655
8,/static/style.css,134.740132


In [66]:
# Option 2: pass as_index=False to groupby — same result
df.groupby("endpoint", as_index=False)["response_ms"].mean()

,endpoint,response_ms
0,/api/auth/login,146.647913
1,/api/auth/logout,155.937153
2,/api/orders,136.009276
3,/api/products,165.849906
4,/api/users,153.003552
5,/health,150.344643
6,/metrics,149.747405
7,/static/app.js,144.265655
8,/static/style.css,134.740132


#### Aggregation with `.agg()`

`.agg()` lets you compute multiple aggregations in a single call. 

Each keyword argument defines one output column:

```python
output_name=(source_column, function)
```

The function can be: `"mean"`, `"count"`, `"sum"`, `"min"`, `"max"`, `"std"` — or any callable.


In [67]:
# Named aggregation with built-in functions 
df.groupby("endpoint").agg(
    request_count=("status_code", "count"),
    avg_response_ms=("response_ms", "mean"),
    max_response_ms=("response_ms", "max"),
).sort_values("request_count", ascending=False)

,request_count,avg_response_ms,max_response_ms
endpoint,,,
/static/style.css,608,134.740132,1185
/metrics,578,149.747405,4928
/api/users,563,153.003552,2496
/health,560,150.344643,1290
/api/auth/login,551,146.647913,1454
/api/auth/logout,541,155.937153,1756
/api/orders,539,136.009276,2451
/api/products,533,165.849906,3073
/static/app.js,527,144.265655,1332


`agg()` also accepts any callable, useful for one-off custom logic that does not have a built-in name.

In [68]:
# Lambda for a custom aggregation — count slow requests per endpoint
df.groupby("endpoint").agg(
    slow_count=("response_ms", lambda x: (x > 300).sum()),
).sort_values("slow_count", ascending=False)

,slow_count
endpoint,
/api/products,76
/health,72
/api/users,69
/api/auth/logout,68
/api/auth/login,64
/static/app.js,62
/metrics,59
/static/style.css,57
/api/orders,40


#### Grouping by multiple columns

Pass a list of column names to group on more than one key. Each unique combination of values forms its own group.


In [69]:
# Group by two columns — pass a list
df.groupby(["method", "status_code"], as_index=False)["response_ms"].mean()

,method,status_code,response_ms
0,DELETE,200,142.098639
1,DELETE,201,130.381818
2,DELETE,301,155.185185
3,DELETE,400,136.973684
4,DELETE,401,112.956522
5,DELETE,404,164.951220
6,DELETE,500,154.000000
7,DELETE,503,99.727273
8,GET,200,147.966190
9,GET,201,139.565217


### Getting Data Out

Pandas is often a step in a pipeline. Here are the exits.


In [70]:
# Back to CSV
df.to_csv("cleaned_access.csv", index=False)

# To JSON
df[["endpoint", "status_code", "response_ms"]].head(10).to_json(
    "summary.json", orient="records", indent=2
)


In [71]:
# Series → Python list
df["endpoint"].tolist()[:5]


['/health', '/api/products', '/static/app.js', '/health', '/api/auth/logout']

In [72]:
# Series → NumPy array
df["response_ms"].to_numpy()[:5]


array([130,  97,  58,  11, 189])

In [73]:
# DataFrame → list of dicts — the shape JSON and most APIs expect
df[["endpoint", "status_code", "response_ms"]].head(3).to_dict(orient="records")


[{'endpoint': '/health', 'status_code': 401, 'response_ms': 130},
 {'endpoint': '/api/products', 'status_code': 200, 'response_ms': 97},
 {'endpoint': '/static/app.js', 'status_code': 201, 'response_ms': 58}]

---
## Section 4 — Q&A

**Topics covered in this notebook:**

- `Series` and `DataFrame` — construction from lists, dicts, arrays
- The Index — labels, gaps after filtering, `reset_index`
- dtypes, memory usage, dtype casting, `category`
- Selection and filtering — `df[]`, `loc`, `iloc`, boolean masks, `query()`
- `value_counts`
- Sorting — single and multi-column
- `ChainedAssignmentError` and copy-on-write
- Missing values — `isnull`, `dropna`, `fillna`, `reset_index`
- Deduplication — `duplicated`, `drop_duplicates`
- Adding rows with `pd.concat`
- `np.where` for derived columns
- `groupby` and `.agg` — including custom aggregations with lambdas
- CSV and JSON I/O; exits via `.tolist()`, `.to_numpy()`, `.to_dict()`